In [1]:
from pathlib import Path
import functools
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import accelforge as af
import functools

import os
af.set_n_parallel_jobs(os.cpu_count(), print_message=True)

### MILESTONE 3
### need to run latency and energy per token on sweep of gamma values 1-12, context length 512-4096 (every 512)
### need to diagnose arch.yaml (baseline) and arch_kv_buffer.yaml (separate KV buffer made) individually

Using 8 parallel jobs


In [2]:
# paths and experiment definitions

ARCHITECTURES = {
    "baseline": Path("designs/arch.yaml"),
    "kv_buffer": Path("designs/arch_kv_buffer.yaml"),
}

WORKLOAD = Path("designs/gpt3_6.7B_kv_cache.yaml")
VALIDATOR = Path("designs/gpt3_175b_kv_cache.yaml")

# Reproducing the old milestone experiments, but for both architectures.

EXPERIMENTS_TO_RUN = [
    {
        "experiment": "all_params_sweep",
        "context_lengths": list(range(512, 4096+128, 512)),
        "gammas": list(range(1,13)),
        "alphas": [round(float(a), 2) for a in np.arange(0.2, 1.0+0.025, 0.025)]
    },    
]

LONG_CSV = Path("milestone3_results_arch_compare_long.csv")
WIDE_CSV = Path("milestone3_results_arch_compare_wide.csv")

for name, path in ARCHITECTURES.items():
    assert path.exists(), f"Missing architecture file for {name}: {path}"

assert WORKLOAD.exists(), f"Missing workload file: {WORKLOAD}"
assert VALIDATOR.exists(), f"Missing validator file: {VALIDATOR}"


In [3]:
# architecture-aware mapping helpers

def _set_keep_all_on_first_memory(spec):
    for node in spec.arch.nodes:
        if isinstance(node, af.arch.Memory):
            node.tensors.keep = "All"
            break

def _set_keep_all_on_main_memory(spec):
    for node in spec.arch.nodes:
        if isinstance(node, af.arch.Memory) and node.name == "MainMemory":
            node.tensors.keep = "All"
            break

@functools.lru_cache(maxsize=None)
def draft_result(arch_name: str, tokens: int):
    spec = af.Spec.from_yaml(
        ARCHITECTURES[arch_name],
        WORKLOAD,
        jinja_parse_data={
            "BATCH_SIZE": 1,
            "N_TOKENS": tokens,
            "N_NEW_TOKENS": 1,
        },
    )
    _set_keep_all_on_first_memory(spec)
    spec.mapper.metrics = af.mapper.Metrics.LATENCY
    return spec.map_workload_to_arch(print_progress=False)

@functools.lru_cache(maxsize=None)
def validator_result(arch_name: str, tokens: int, lookahead: int):
    spec = af.Spec.from_yaml(
        ARCHITECTURES[arch_name],
        VALIDATOR,
        jinja_parse_data={
            "BATCH_SIZE": lookahead,
            "N_TOKENS": tokens + lookahead,
            "N_NEW_TOKENS": 1,
        },
    )
    _set_keep_all_on_first_memory(spec)
    spec.mapper.metrics = af.mapper.Metrics.LATENCY
    return spec.map_workload_to_arch(print_progress=False)

@functools.lru_cache(maxsize=None)
def true_validator_result(arch_name: str, tokens: int, lookahead: int):
    spec = af.Spec.from_yaml(
        ARCHITECTURES[arch_name],
        VALIDATOR,
        jinja_parse_data={
            "BATCH_SIZE": 1,
            "N_TOKENS": tokens + lookahead,
            "N_NEW_TOKENS": lookahead,
        },
    )
    _set_keep_all_on_main_memory(spec)
    spec.mapper.metrics = af.mapper.Metrics.LATENCY
    return spec.map_workload_to_arch(print_progress=False)

def expected_tokens_per_round(gamma: int, alpha: float) -> float:
    if alpha >= 1.0:
        return gamma + 1
    return (1 - alpha ** (gamma + 1)) / (1 - alpha)

@functools.lru_cache(maxsize=None)
def baseline_per_token(arch_name: str, tokens: int, gamma: int):
    total_energy = 0.0
    total_latency = 0.0

    for i in range(gamma):
        result = validator_result(arch_name, tokens + i, 1)
        total_energy += float(result.energy())
        total_latency += float(result.latency())

    return total_energy / gamma, total_latency / gamma

@functools.lru_cache(maxsize=None)
def speculative_round_cost(arch_name: str, context_len: int, gamma: int):
    round_energy = 0.0
    round_latency = 0.0

    for i in range(gamma):
        result = draft_result(arch_name, context_len + i)
        round_energy += float(result.energy())
        round_latency += float(result.latency())

    verify_result = true_validator_result(arch_name, context_len, gamma)
    round_energy += float(verify_result.energy())
    round_latency += float(verify_result.latency())

    return round_energy, round_latency

def speculative_per_token(arch_name: str, context_len: int, gamma: int, alpha: float):
    round_energy, round_latency = speculative_round_cost(arch_name, context_len, gamma)
    expected_yield = expected_tokens_per_round(gamma, alpha)
    return round_energy / expected_yield, round_latency / expected_yield


In [4]:
# rerun all old experiments on both architectures and save one consolidated CSV

draft_result.cache_clear()
validator_result.cache_clear()
true_validator_result.cache_clear()
baseline_per_token.cache_clear()
speculative_round_cost.cache_clear()

rows = []

for exp_cfg in EXPERIMENTS_TO_RUN:
    exp_name = exp_cfg["experiment"]
    for arch_name in ARCHITECTURES:
        for ctx in exp_cfg["context_lengths"]:
            for gamma in exp_cfg["gammas"]:
                base_energy_per_tok, base_latency_per_tok = baseline_per_token(
                    arch_name, ctx, gamma
                )

                for alpha in exp_cfg["alphas"]:
                    spec_energy_per_tok, spec_latency_per_tok = speculative_per_token(
                        arch_name, ctx, gamma, alpha
                    )

                    row = {
                        "experiment": exp_name,
                        "arch_name": arch_name,
                        "arch_path": str(ARCHITECTURES[arch_name]),
                        "context_len": ctx,
                        "gamma": gamma,
                        "alpha": alpha,
                        "base_energy_per_tok": base_energy_per_tok,
                        "base_latency_per_tok": base_latency_per_tok,
                        "spec_energy_per_tok": spec_energy_per_tok,
                        "spec_latency_per_tok": spec_latency_per_tok,
                        "latency_speedup": base_latency_per_tok / spec_latency_per_tok,
                        "energy_ratio": spec_energy_per_tok / base_energy_per_tok,
                    }
                    rows.append(row)

                    print(
                        f"{exp_name:>15} | arch={arch_name:<9} | "
                        f"ctx={ctx:<4} | gamma={gamma:<2} | alpha={alpha:.2f}"
                    )

                    if len(rows) % 10 == 0:
                        pd.DataFrame(rows).to_csv(LONG_CSV, index=False)

long_df = pd.DataFrame(rows).sort_values(
    ["experiment", "arch_name", "context_len", "gamma", "alpha"]
)
long_df.to_csv(LONG_CSV, index=False)

print(f"Saved consolidated long-form results to {LONG_CSV}")
long_df.head()


NameError: name 'EXPERIMENTS' is not defined

In [ ]:
# Cell 4: make an architecture-comparison CSV with baseline-vs-kv-buffer deltas

long_df = pd.read_csv(LONG_CSV)

key_cols = ["experiment", "context_len", "gamma", "alpha"]
value_cols = [
    "base_energy_per_tok",
    "base_latency_per_tok",
    "spec_energy_per_tok",
    "spec_latency_per_tok",
    "latency_speedup",
    "energy_ratio",
]

wide_df = (
    long_df.pivot_table(
        index=key_cols,
        columns="arch_name",
        values=value_cols,
        aggfunc="first",
    )
    .sort_index(axis=1)
)

wide_df.columns = [f"{metric}_{arch}" for metric, arch in wide_df.columns]
wide_df = wide_df.reset_index()

wide_df["base_latency_speedup_kv_buffer_vs_baseline_arch"] = (
    wide_df["base_latency_per_tok_baseline"] / wide_df["base_latency_per_tok_kv_buffer"]
)
wide_df["base_energy_ratio_kv_buffer_vs_baseline_arch"] = (
    wide_df["base_energy_per_tok_kv_buffer"] / wide_df["base_energy_per_tok_baseline"]
)
wide_df["spec_latency_speedup_kv_buffer_vs_baseline_arch"] = (
    wide_df["spec_latency_per_tok_baseline"] / wide_df["spec_latency_per_tok_kv_buffer"]
)
wide_df["spec_energy_ratio_kv_buffer_vs_baseline_arch"] = (
    wide_df["spec_energy_per_tok_kv_buffer"] / wide_df["spec_energy_per_tok_baseline"]
)

wide_df.to_csv(WIDE_CSV, index=False)

print(f"Saved architecture-comparison results to {WIDE_CSV}")
wide_df.head()
